#Financial Product Complaint Classification and Summarization

## **Business Context**

### **Description**
*In the modern financial industry, customer complaints play a crucial role in identifying areas where financial institutions can improve their services. Effectively categorizing these complaints into specific product categories, such as credit reports, student loans, or money transfers, is essential for addressing customer concerns promptly by routing the tickets to relevant personnel. Leveraging Generative AI for text classification can help financial institutions better understand customer grievances and respond more efficiently. Apart from this, a summary of the customer complaint helps the support personnel quickly grasp the gist of the grievance*

### **Objective**
*The primary goal of this project is to utilize Generative AI techniques to improve the classification and summarization of customer complaints in the financial sector.
Specifically, the project will focus on:*

1. **Text-to-Label Classification:** *Implementing Zero-shot and Few-shot prompting methods to accurately classify customer complaints into relevant product categories.*
2. **Text-to-Text Summarization:** *Using Zero-shot prompting to generate concise summaries of customer complaints, enabling more personalized and effective responses.*


# **Section 1 : Setting Up for Prompt Engineering with Mistral Model**

### **Install & Importing neccessary libraries**

In [5]:
!apt-get update
!apt-get install -y ninja-build cmake
!pip install ipywidgets --upgrade


0% [Working]
            
Hit:1 https://cli.github.com/packages stable InRelease

0% [Connecting to archive.ubuntu.com] [Connecting to security.ubuntu.com (91.18
                                                                               
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:7 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:9 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Hit:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Reading package lists... Done
W: Skipping acq

In [6]:
import torch

In [7]:
# This part of code will skip all the un-necessary warnings which can occur during the execution of this project.
import warnings
warnings.filterwarnings("ignore", category=Warning)

In [8]:
# Installation for GPU llama-cpp-python==0.2.69
!CMAKE_ARGS="-DLLAMA_CUDA=on" pip install llama-cpp-python==0.2.69
# For downloading the models from HF Hub
!pip install huggingface_hub

In [9]:
!pip install evaluate #HF to evaluate metrics for ML models
!pip install bert-score #how similar 2 sentences are using contextual embeddings from BERT like models

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 6.2 MB/s eta 0:00:00


In [10]:
!pip freeze > requirement.txt

In [11]:
# Basic Imports for Libraries
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

import pandas as pd
import numpy as np
from tqdm import tqdm
import json
import re

import torch
import evaluate

# from google.colab import drive
import locale

###1: Importing Libaries and Mistral Model

In [12]:
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

In [13]:
model_name_or_path = "TheBloke/Mistral-7B-Instruct-v0.2-GGUF"
model_basename = "mistral-7b-instruct-v0.2.Q5_K_M.gguf"

In [14]:
model_path = hf_hub_download(
    repo_id=model_name_or_path,
    filename=model_basename
    )

mistral-7b-instruct-v0.2.Q5_K_M.gguf:   0%|          | 0.00/5.13G [00:00<?, ?B/s]

In [15]:
lcpp_llm = Llama(
        model_path=model_path, #GGUF model file # mistral-7b
        n_threads=2,  # no CPU cores used
        n_batch=512,  # tokens processed per computation step
        n_gpu_layers=43,  # how many model layers are offloaded to GPU
        n_ctx=4096,  # context window size
    )

llama_model_loader: loaded meta data with 24 key-value pairs and 291 tensors from /root/.cache/huggingface/hub/models--TheBloke--Mistral-7B-Instruct-v0.2-GGUF/snapshots/3a6fbf4a41a1d52e415a4958cde6856d34b2db93/mistral-7b-instruct-v0.2.Q5_K_M.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.name str              = mistralai_mistral-7b-instruct-v0.2
llama_model_loader: - kv   2:                       llama.context_length u32              = 32768
llama_model_loader: - kv   3:                     llama.embedding_length u32              = 4096
llama_model_loader: - kv   4:                          llama.block_count u32              = 32
llama_model_loader: - kv   5:                  llama.feed_forward_length u32              = 14336
llama_model_loa

# **Text to Label generation**

In [46]:
system_message = """
You are an expert financial complaint classification assistant.
Your task is to read a customer's complaint narrative and classify it into the correct product category.

Choose only one category from the following list (return exactly as written):
credit_card, retail_banking, credit_reporting, mortgages_and_loans, debt_collection.

Return only the category name. Do not provide explanations.
"""

In [47]:
zero_shot_prompt_template = """<s>[INST]
<<SYS>>
{system_message}
<</SYS>>

Customer Complaint:
{user_input}

Product Category:
[/INST]

"""

In [18]:
def generate_prompt(system_message,user_input):
    prompt=zero_shot_prompt_template.format(system_message=system_message,user_input=user_input) #zero shot prompt
    return prompt

In [19]:
def generate_mistral_response(input_text):

    # Combine user_prompt and system_message to create the prompt
    prompt = generate_prompt(system_message,input_text)

    # Define the Llama model along with its parameters for generating a response
    response = lcpp_llm(
        prompt=prompt,
        max_tokens=1200,
        temperature=0,
        top_p = 0.95,
        repeat_penalty=1.2,
        top_k=50,
        stop=["</s>"],
        echo=False
    )

    # Extract and return the response text
    response_text = response["choices"][0]["text"].strip()
    print(response_text)
    return response_text

In [21]:
data = pd.read_csv("/content/Complaints_classification.csv")

In [22]:
# Randomly select 30 rows
new_data = data.sample(n=30, random_state=40)

In [23]:
print(new_data.head())

              product                                          narrative  \
167    retail_banking  fraudulent charge totaling made capital one ch...   
169  credit_reporting  block except otherwise provided section consum...   
461       credit_card  usaa master plan collect cancellation debt usa...   
253  credit_reporting  block except otherwise provided section consum...   
42   credit_reporting  open account acct opened balance account acct ...   

                                               summary  
167  A fraudulent charge was made on the individual...  
169  The text outlines various stipulations regardi...  
461  The input appears to be a complaint about USAA...  
253  The text pertains to the stipulations and oper...  
42   The input is about various accounts being open...  


In [24]:
#create a new column DF 'mistral_response' & populate it with responses generated by applying the 'general_mistral_response' function to each 'narrative' in the DF & prepare the mistral_response_cleaned col using extract_category function
new_data['mistral_response'] = new_data['narrative'].apply(lambda x: generate_mistral_response(x))



llama_print_timings:        load time =     649.71 ms
llama_print_timings:      sample time =       3.97 ms /     6 runs   (    0.66 ms per token,  1510.19 tokens per second)
llama_print_timings: prompt eval time =     649.20 ms /   338 tokens (    1.92 ms per token,   520.64 tokens per second)
llama_print_timings:        eval time =     139.62 ms /     5 runs   (   27.92 ms per token,    35.81 tokens per second)
llama_print_timings:       total time =     817.65 ms /   343 tokens
Llama.generate: prefix-match hit


credit_card.



llama_print_timings:        load time =     649.71 ms
llama_print_timings:      sample time =       3.17 ms /     6 runs   (    0.53 ms per token,  1893.34 tokens per second)
llama_print_timings: prompt eval time =     513.79 ms /   512 tokens (    1.00 ms per token,   996.52 tokens per second)
llama_print_timings:        eval time =     150.38 ms /     5 runs   (   30.08 ms per token,    33.25 tokens per second)
llama_print_timings:       total time =     684.44 ms /   517 tokens
Llama.generate: prefix-match hit


credit_reporting



llama_print_timings:        load time =     649.71 ms
llama_print_timings:      sample time =       3.27 ms /     6 runs   (    0.54 ms per token,  1835.42 tokens per second)
llama_print_timings: prompt eval time =     320.88 ms /   189 tokens (    1.70 ms per token,   589.01 tokens per second)
llama_print_timings:        eval time =     146.58 ms /     5 runs   (   29.32 ms per token,    34.11 tokens per second)
llama_print_timings:       total time =     488.83 ms /   194 tokens
Llama.generate: prefix-match hit


...

credit_reporting



llama_print_timings:        load time =     649.71 ms
llama_print_timings:      sample time =       3.23 ms /     6 runs   (    0.54 ms per token,  1855.86 tokens per second)
llama_print_timings: prompt eval time =     523.28 ms /   512 tokens (    1.02 ms per token,   978.45 tokens per second)
llama_print_timings:        eval time =     155.76 ms /     5 runs   (   31.15 ms per token,    32.10 tokens per second)
llama_print_timings:       total time =     699.21 ms /   517 tokens
Llama.generate: prefix-match hit


credit_reporting



llama_print_timings:        load time =     649.71 ms
llama_print_timings:      sample time =       3.26 ms /     6 runs   (    0.54 ms per token,  1840.49 tokens per second)
llama_print_timings: prompt eval time =     212.12 ms /   102 tokens (    2.08 ms per token,   480.86 tokens per second)
llama_print_timings:        eval time =     158.42 ms /     5 runs   (   31.68 ms per token,    31.56 tokens per second)
llama_print_timings:       total time =     389.76 ms /   107 tokens


credit_reporting


In [25]:
new_data['mistral_response']

,mistral_response
167,credit_card.
169,credit_reporting
461,debt_collection.
253,credit_reporting
42,retail_banking
369,credit_reporting
26,mortgages_and_loans
377,credit_reporting
238,credit_reporting
374,credit_reporting


In [26]:
#Extracting a category label from text using regular expressions.
#Find a structured label first. If that fails, it falls back to searching for known category keywords.

#Creates a function that takes 1 piece of text as input
def extract_category(text):
    # Define the regex pattern to match "category:" or "Category:" followed by a word
    pattern = r'category:\s*(\w+)'

    # Use re.search with the re.IGNORECASE flag to make it case-insensitive
    match = re.search(pattern, text, re.IGNORECASE)

    # If a match is found, return the captured group, else return None
    if match:
        return match.group(1)
    else:
        pattern1 = r'(credit_card|retail_banking|credit_reporting|mortgages_and_loans|debt_collection)'
        match = re.search(pattern1, text, re.IGNORECASE)
        if match:
            return match.group()
        else:
            return ''

In [27]:
#Handles missing responses
#Extract structured labels like "Category: Credit_Card"
#Fall back to keyword search
#Defaults to "other" when nothing matches

import re

def extract_category(text):
    if not text:
        return "Other"

    # Match "Category: Something With Spaces"
    pattern = r'category:\s*([A-Za-z\s]+)'
    match = re.search(pattern, text, re.IGNORECASE)

    if match:
        return match.group(1).strip()

    # Fallback to known labels
    allowed = [
        "credit_card",
        "retail_banking",
        "credit_reporting",
        "mortgages_and_loans",
        "debt_collection",
        "Other"
    ]

    for cat in allowed:
        if cat.lower() in text.lower():
            return cat

    return "Other"

In [28]:
new_data['mistral_response_cleaned'] = new_data['mistral_response'].apply(lambda x: extract_category(x))

In [29]:
new_data.head()

,product,narrative,summary,mistral_response,mistral_response_cleaned
167,retail_banking,fraudulent charge totaling made capital one ch...,A fraudulent charge was made on the individual...,credit_card.,credit_card
169,credit_reporting,block except otherwise provided section consum...,The text outlines various stipulations regardi...,credit_reporting,credit_reporting
461,credit_card,usaa master plan collect cancellation debt usa...,The input appears to be a complaint about USAA...,debt_collection.,debt_collection
253,credit_reporting,block except otherwise provided section consum...,The text pertains to the stipulations and oper...,credit_reporting,credit_reporting
42,credit_reporting,open account acct opened balance account acct ...,The input is about various accounts being open...,retail_banking,retail_banking


In [30]:
f1 =  f1_score(new_data['product'], new_data['mistral_response_cleaned'],average='micro')
print(f'F1 Score: {f1}')

F1 Score: 0.8333333333333334


Since micro-averaging aggregates all true positives, false positives ,and false negatives globally, this score is equivalent to overall accuracy in a multi-class setting.

In [ ]:
f2 = f1_score(
    new_data['product'],
    new_data['mistral_response_cleaned'],
    average='micro'
)

print(f'F1 Score: {f2}')

F1 Score: 0.8333333333333334


F1 score for both mistral_response and mistral_response_cleaned is 0.8333, indicating that the cleaning step did not change the model's classification and performance.

The cleaning function mainly standardizes the output format of the model predictions (eg, extracting the category label and normalizing text). However, since the predicted categories remained the same before and after cleaning, the number of true positives, false positives, and false negative did not change.

Because the evaluation uses micro-average F1 score, which measures overall classification accuracy across all classes, the resulting score remains identical.

Therefore, the cleaning step improves the consistency of prediction labels but does not affect the model's predicitive performance in this dataset.